# MediaVault · серверный импорт в Google Drive

Видео скачивается **на сервер Google Colab**, собирается в MP4, шифруется AES-GCM
и загружается в ваш Drive. Телефон передаёт небольшое задание; видеотрафик при
импорте не идёт через телефон, GitHub Pages или Cloudflare Worker.

1. На сайте MediaVault откройте свою библиотеку → **Импорт**. Укажите публичные
   ссылки, коллекцию и серии, нажмите **Подготовить серверный импорт → Скопировать задание**.
2. Здесь выберите **Среда выполнения → Выполнить всё**. Достаточно бесплатного CPU;
   GPU, Pro, банковская карта и Cloudflare не требуются.
3. Вставьте скопированное задание в появившееся поле. Затем войдите в аккаунт
   владельца и введите фразу библиотеки в скрытом поле `getpass`.
4. После завершения нажмите **Обновить библиотеку** на сайте.

Используется собственный разбор публичного HTML и открытые адаптеры плееров
`anime-dl-core`. API аниме-сайтов, DRM и платный доступ не используются.
Поддерживаются найденные публичные MP4/HLS и плееры Kodik, Aniboom, Sibnet,
Animedia. Если сайт отдаёт только проверку браузера или скрывает список серий,
понадобятся отдельные публичные ссылки; поддержка всех зеркал не гарантируется.

Бесплатные сессии Colab имеют переменную длительность и могут отключаться.
После каждой завершённой серии результат сохраняется в Drive. Повторный запуск
того же задания пропустит уже записанные серии. Временные файлы занимают место
только для текущей серии и удаляются после неё. Пока идёт импорт, не изменяйте
библиотеку с другого устройства. В Drive должно хватать свободного места.

**Не вставляйте пароль Google, Client Secret, токены или секретную фразу в JSON
задания или ячейки кода.** Фраза вводится только в скрытом поле при запуске и
не записывается в блокнот. Не делитесь работающей сессией и её выводом.

In [ ]:
# 1. Подготовка среды. Код загружается и проверяется до входа в Google.
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, urllib.request

SOURCE_HASHES = {'server/source_parser.py': '679ad65bd1c3193e99246d7aa9fbb3574bb11dfbbdbc83aca057cfe4eb18ebd6', 'server/media_download.py': 'c8da59fe3b6d58e2828bd1096d2d5586ddc23e9f601bd5056773391931a6b13a', 'server/run_import.py': '14010cfae588ecfd7833ac48cce4d26916018bda24e206e35586941c0f8ea74d', 'server/vault_drive.py': '4b3a710d1f3e31b4c58ec1d2c35e4b406be435cff56ae46808a184ceee161ad5', 'server/player_catalog.py': '4620118ed17601278c85060015e77704cc97655e3cea1730763e6bfb37d07966', 'server/player_compat.py': 'a694035e4e845455f6c83b72e0fd4cd3c8c6b91fd99b4db7c9f5887283744987', 'server/browser_player.py': '74f45fd3b66948a3e9524487be945f533df4e71cfab81fbfe00ebedb35c543cd', 'server/public_page_data.py': 'c4e04469ac848e0dc5bee3434568df7287d84802d7b1b71117906cf3d8d11349', 'server/requirements.txt': 'ae12846354093174518a697aae503d46973c6a7b2c0420c25da6a13eec8fdc68', 'server/title_extras.py': '252b4c4cfc71b80d6b1a413d257bdad2a5724fe2bdfc8586cc903fa01a8d9636', 'server/manga_sources.py': '9f645598baf601bdc6623688b58d7d96cbbc67aa31cff8e4d43df84fb6e0b046', 'server/manga_raw_adapters.py': 'c2a6497ef80ddb3bb28b58cec32cc4068bcbabcda244447a388e3ad43598ee1b', 'server/manga_webtoons.py': '23f003e369d15127e93b51e1ce57b4a46971a8762defbd8881b15b9ad4a29014', 'server/manga_catalog_adapters.py': 'c1185db6741ad59f1b4281917db13679c33e936a2f3248179b33303e4c2378d1', 'server/manga_download.py': '642a3c17a94f89cda0d69837f2fe5eade46e58444e7324b3ce834d5bda842e35', 'server/title_identity.py': 'd52c1cd981479ca5021d7a08431e9254b2d6313caef13156ba59496abade350e', 'server/suwayomi_bridge.py': '22c40b4488c34d00e6596a443c3104ee2174eefa4d823e63c87d326ba05765ea'}
BASE = "https://raw.githubusercontent.com/Onikri/MediaVault/main/"
RUNTIME = Path("/content/mediavault-server")
RUNTIME.mkdir(parents=True, exist_ok=True)
for relative, expected in SOURCE_HASHES.items():
    with urllib.request.urlopen(BASE + relative, timeout=45) as response:
        data = response.read(2 * 1024 * 1024)
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError("Исходный код обновился. Откройте свежий блокнот по ссылке с сайта.")
    (RUNTIME / Path(relative).name).write_bytes(data)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", str(RUNTIME / "requirements.txt")], check=True)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
sys.path.insert(0, str(RUNTIME))
from run_import import validate_job, run_job
from vault_drive import GoogleDriveAdapter, VaultDrive, VaultError
from source_parser import SourceError
print("Среда готова. Платные ресурсы не включались.")

In [ ]:
# 2. Вставь задание кнопкой «Скопировать задание» на сайте (оно копируется одной строкой).
# Здесь нужны только ссылки и ID каталога. Фразу библиотеки сюда вводить нельзя.
job_text = input("Задание MediaVault (JSON): ").strip()
try:
    job = json.loads(job_text)
    validate_job(job)
except Exception:
    job_text = ""
    raise RuntimeError("Некорректное задание. Скопируйте его кнопкой на сайте заново.") from None
job_text = ""
print("Задание проверено. Далее откроется вход в Google.")

In [ ]:
# 3. Вход владельца, открытие библиотеки и серверный импорт.
from google.colab import auth
from getpass import getpass

auth.authenticate_user()
vault = None
phrase = None
try:
    # Адрес Google-аккаунта проверяется до любых записей в Drive.
    vault = VaultDrive(GoogleDriveAdapter.from_adc())
    phrase = getpass("Секретная фраза открытой библиотеки: ")
    try:
        vault.unlock(job["library_id"], phrase)
    except Exception:
        raise VaultError("Не удалось открыть библиотеку. Проверьте ID, аккаунт и секретную фразу.") from None
    finally:
        phrase = None
    result = run_job(job, vault)
    print("Откройте сайт и нажмите «Обновить библиотеку»:")
    print("https://onikri.github.io/MediaVault/?library=" + vault.library_id)
except (SourceError, VaultError) as error:
    print("Импорт остановлен:", str(error))
    print("Завершённые серии сохранены. После исправления причины можно запустить то же задание.")
except KeyboardInterrupt:
    print("Импорт прерван. Завершённые серии сохранены; то же задание можно продолжить.")
except Exception:
    print("Импорт остановлен из-за ошибки сети, доступа Google или квоты.")
    print("Проверьте свободное место и аккаунт. Повторите запуск; готовые серии будут пропущены.")
finally:
    phrase = None
    if vault is not None:
        vault.close()
    vault = None

### Что сохраняется

Drive содержит каталог, зашифрованные блоки по 4 МиБ и зашифрованные манифесты
восстановления. Разделы называются «Аниме», «Манга», «Ранобэ», «Книги», «Музыка»; имена тайтлов и
видео находятся внутри зашифрованного каталога, а папки тайтлов, сезонов/томов и файлов имеют случайные ID.
Совместимость с веб-приложением проверяется тестами Python ↔ Web Crypto.

Фраза и производный ключ используются в памяти текущего процесса. Авторизацией
Google управляет Colab. Завершите сессию через **Среда выполнения → Отключиться
и удалить среду выполнения**, когда импорт закончен. Во время обычного просмотра
на сайте видео, разумеется, скачивается из Drive на устройство зрителя.

[Исходный код и инструкции](https://github.com/Onikri/MediaVault)
· [Использованные открытые библиотеки](https://github.com/Onikri/MediaVault/blob/main/THIRD_PARTY.md)